# Pediatric Neuro-Oncology Surgical Planning Agent — Quickstart

Upload `pediatric-neuro-oncology-agent.zip`, install dependencies, run text demo, optional DICOM/NIfTI advanced imaging, and autonomous evidence refresh.


In [ ]:
from google.colab import files
uploaded = files.upload()


In [ ]:
import glob, zipfile, os
zips = glob.glob('/content/*pediatric-neuro-oncology-agent*.zip')
assert zips, 'No project zip found.'
with zipfile.ZipFile(zips[0]) as z:
    z.extractall('/content')
os.chdir('/content/pediatric-neuro-oncology-agent')
print('cwd =', os.getcwd())


In [ ]:
!pip install -q -r requirements.txt
print('installed')


In [ ]:
import os, getpass
os.environ['NVIDIA_API_KEY'] = getpass.getpass('NVIDIA_API_KEY, press Enter for MOCK mode: ')
os.environ['NEMOTRON_MODEL'] = 'nvidia/nemotron-3-super-120b-a12b'
os.environ.setdefault('NVIDIA_BASE_URL', 'https://integrate.api.nvidia.com/v1')
print('Nemotron model:', os.environ['NEMOTRON_MODEL'])


In [ ]:
!python run_demo.py sample_cases/case_003_diffuse_midline_glioma.txt


In [ ]:
import glob, os
mds = glob.glob('outputs/*.md')
latest = max(mds, key=os.path.getmtime)
print(latest)
print(open(latest, encoding='utf-8').read()[:4000])


## Optional: DICOM / NIfTI advanced imaging
Upload a DICOM zip/folder contents or `.nii/.nii.gz` files. The module produces structured imaging findings and `outputs/surgical_risk_map.jpg`.


In [ ]:
import os, zipfile, shutil, glob
from pathlib import Path
from google.colab import files
os.makedirs('medical_inputs', exist_ok=True)
up = files.upload()
for fname in up.keys():
    if fname.lower().endswith('.zip'):
        with zipfile.ZipFile(fname) as z:
            z.extractall('medical_inputs')
    else:
        shutil.move(fname, str(Path('medical_inputs') / Path(fname).name))
print('medical_inputs preview:')
for p in glob.glob('medical_inputs/**/*', recursive=True)[:30]:
    print(p)


In [ ]:
from advanced_medical_imaging import analyze_medical_study
case = {'age': 8, 'tumor_type': 'diffuse midline glioma consideration', 'tumor_location': 'pons', 'pathology': 'H3 K27-altered pending confirmation', 'symptoms': 'diplopia, gait imbalance'}
res = analyze_medical_study('medical_inputs', structured_case=case, output_dir='outputs')
print(res.get('imaging_description_text', ''))
print('Risk map:', res.get('risk_map_path'))


In [ ]:
from IPython.display import Image, display
import os
if os.path.exists('outputs/surgical_risk_map.jpg'):
    display(Image(filename='outputs/surgical_risk_map.jpg'))


## Optional: autonomous evidence refresh


In [ ]:
!python autonomous_refresh_loop.py --once --pubmed-days 30 --pubmed-retmax 30
!tail -n 20 logs/autonomous_runs.log
